# UC4 — Executive Business Intelligence + AI Lead Scoring

---

## What this notebook does

This notebook covers two business intelligence use cases in one place:

### Part A — Executive Business Intelligence (3 domains)
Reads aggregated KPIs from the Gold Materialized Views and uses an LLM to generate
plain-English executive synthesis — not data restating, but root-cause explanation.

| Domain | Source table | Business problem addressed |
|---|---|---|
| Revenue performance | mv_revenue_by_region | Revenue audit — which regions are growing and why |
| Vendor return rates | mv_return_rate_by_vendor | Returns fraud — which vendors need action |
| Slow-moving inventory | mv_slow_moving_products | Inventory blindspot — what is sitting unsold |

### Part B — AI Lead Scoring
Scores every customer on purchase value, frequency, recency, and return behaviour.
Labels each customer HOT / WARM / COLD. Uses `ai_query()` — Databricks' SQL-native
Foundation Model function — to generate a one-sentence SHAP-style explanation per
customer naming the top factor that drove their score.

---

## Output tables

| Table | Contents |
|---|---|
| `globalmart.gold.ai_business_insights` | 3 executive summaries, one per domain |
| `globalmart.gold.customer_lead_scores` | One row per customer with score + SHAP explanation |
| `globalmart.gold.pipeline_run_log` | One row per run with metadata |

---

## Design rules

- **Never pass raw rows to the LLM.** Always aggregate first. 5 KPI values produce better
  synthesis than 5,000 rows — and will not hit context limits.
- **CATALOG constant** — all table references use `f"{CATALOG}.table_name"`. Change
  the catalog in one place and it updates everywhere.
- **ai_query()** is used for lead scoring explanations. It calls the Foundation Model
  directly inside SQL — no Python API client needed for that section.
- **OPTIMIZE + ZORDER** runs after every write to the two largest output tables.

In [0]:
# openai is not pre-installed on Databricks clusters.
# Run this cell first. The kernel restarts automatically so the package
# is available in every cell that follows.
# On subsequent runs within the same cluster session, skip this cell.

%pip install openai
dbutils.library.restartPython()

In [0]:
# All imports and constants are defined here so any reader can see the full
# dependency list without hunting through the notebook.
#
# Key constants:
#   CATALOG    — all Gold tables live here. Change once to move the catalog.
#   MODEL_NAME — the Foundation Model used for both the Python LLM calls
#                (executive summaries) and the ai_query() SQL calls (lead scoring).
#   NOTEBOOK_NAME — written to the run log for operational tracking.
#
# LLM connection:
#   The OpenAI SDK is pointed at the Databricks AI Gateway.
#   The token is pulled automatically from the active session — no hardcoded
#   credentials anywhere in this notebook.

import time
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType
from openai import OpenAI

# Connect to the Databricks AI Gateway
# databricks-gpt-oss-20b is available on the Free Edition at no cost
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1"
)

MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC4_Executive_Business_Intelligence"
CATALOG       = "globalmart.gold"   # change here only if catalog moves

print(f"Model     : {MODEL_NAME}")
print(f"Catalog   : {CATALOG}")
print(f"Run started: {datetime.now().isoformat()}")

In [0]:
# Three helper functions used by every Python LLM call in this notebook.
# These are NOT used by ai_query() — that runs entirely inside SQL.

def extract_text(response) -> str:
    """
    Pull only the answer text from the model response.

    Why this is needed:
    databricks-gpt-oss-20b returns a JSON array, not a plain string:
      [{"type": "reasoning", ...},        <- internal chain-of-thought, discard
       {"type": "text", "text": "..."}]   <- the actual answer, keep this

    Without this function the raw reasoning chain appears alongside the
    executive summary, making the output unreadable.
    """
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    """
    Call the LLM with automatic retry on transient failures.

    Exponential backoff: waits 2s, 4s, 6s between attempts.
    On total failure returns 'LLM_UNAVAILABLE: <error>' so the pipeline
    continues and the failure is recorded rather than crashing the notebook.
    Temperature 0.4 allows natural narrative variation in executive prose.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.4
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_output(text: str, min_words: int = 40) -> dict:
    """
    Quality-check the LLM output before writing to the Gold layer.

    Catches three failure modes:
      too_short        — response was truncated (under 40 words)
      llm_call_failed  — all retries exhausted (starts with LLM_UNAVAILABLE)
      refusal_detected — model refused instead of answering

    Returns a dict:
      text             — the output text
      llm_check        — PASS or REVIEW
      llm_check_detail — comma-separated issues, or 'none'

    REVIEW outputs are still written to Gold but flagged for human audit.
    """
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable"]
    issues = []
    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(phrase in text.lower() for phrase in refusal_phrases):
        issues.append("refusal_detected")
    return {
        "text":             text,
        "llm_check":        "PASS" if not issues else "REVIEW",
        "llm_check_detail": ", ".join(issues) if issues else "none"
    }


# System message for all three executive summary domains.
# The model's job is to explain WHAT numbers mean — root cause, not restatement.
# No company name, no bullet points, no markdown.
SYSTEM_MSG_EXEC = (
    "You are a senior business analyst writing executive synthesis for a national retail chain. "
    "Your job is to explain what the numbers mean, not restate them. "
    "Identify the root cause behind each metric, name the specific regions or vendors involved, "
    "and end with a clear recommended action the leadership team should take. "
    "Write in plain, authoritative business English. "
    "4-6 sentences of connected prose. No bullet points, no markdown, no headers, no company name."
)

print("LLM helpers defined.")

In [0]:
# This notebook reads Gold only — it does NOT touch Bronze or Silver.
#
# Materialized Views (pre-aggregated by the Gold pipeline, fast reads):
#   mv_revenue_by_region       columns: year, month, region, total_sales, order_count
#   mv_return_rate_by_vendor   columns: vendor_id, vendor_name, return_rate_pct,
#                                        total_returns, total_orders_sold
#   mv_slow_moving_products    columns: product_id, region, total_sales,
#                                        days_since_last_sale, total_quantity_sold
#
# Fact and dimension tables:
#   fact_orders    — one row per order, with customer_id and timestamps
#   fact_returns   — one row per return event, with customer_id and refund_amount
#   fact_sales     — one row per order-product line item, with sales and order_date_key
#   dim_vendors    — vendor_id, vendor_name
#   dim_customers  — customer_id, customer_name, segment, region

df_revenue   = spark.table(f"{CATALOG}.mv_revenue_by_region")
df_vendor_rr = spark.table(f"{CATALOG}.mv_return_rate_by_vendor")
df_slow      = spark.table(f"{CATALOG}.mv_slow_moving_products")
df_orders    = spark.table(f"{CATALOG}.fact_orders")
df_returns   = spark.table(f"{CATALOG}.fact_returns")
df_vendors   = spark.table(f"{CATALOG}.dim_vendors")

# Also load these two for Part B — AI Lead Scoring
df_sales     = spark.table(f"{CATALOG}.fact_sales")
df_customers = spark.table(f"{CATALOG}.dim_customers")

print("Gold tables loaded:")
print(f"  mv_revenue_by_region      : {df_revenue.count():,} rows")
print(f"  mv_return_rate_by_vendor  : {df_vendor_rr.count():,} rows")
print(f"  mv_slow_moving_products   : {df_slow.count():,} rows")
print(f"  fact_orders               : {df_orders.count():,} rows")
print(f"  fact_returns              : {df_returns.count():,} rows")
print(f"  fact_sales                : {df_sales.count():,} rows")
print(f"  dim_vendors               : {df_vendors.count():,} rows")
print(f"  dim_customers             : {df_customers.count():,} rows")

---
## Part A — Executive Business Intelligence

The three cells below each follow the same pattern:
1. **KPI Aggregation** — compute 5 summary statistics from the MV, serialise to JSON
2. **LLM Call** — pass the JSON to the model with a prompt asking for root-cause synthesis

The model is never given raw rows. The KPI JSON is also stored in the output table
so any analyst can verify exactly what data produced each summary.

In [0]:
# PURPOSE: Compute the 5 KPIs passed to the LLM for the revenue summary.
#
# mv_revenue_by_region column reference (from gold.py):
#   year, quarter, month, month_name, region
#   total_sales, total_profit, line_item_count
#   order_count, avg_line_item_value, avg_order_value
#
# The 'month' column is an integer (1-12), not a date string.
# 'latest_month' finds the highest month integer in the table.
#
# MoM delta formula:
#   (current_sales - prior_cumulative_sales) / prior_cumulative_sales * 100
#   If a region has no prior data, delta defaults to 0.

# Step 1: find the most recent reporting period
latest_month = df_revenue.agg(F.max("month")).collect()[0][0]

# Step 2: current period — sales and order count per region
revenue_current = (
    df_revenue
    .filter(F.col("month") == latest_month)
    .groupBy("region")
    .agg(
        F.sum("total_sales").alias("total_sales"),
        F.sum("order_count").alias("order_count")
    )
    .orderBy(F.col("total_sales").desc())
    .collect()
)

# Step 3: prior periods — cumulative sales per region for MoM comparison
prior_by_region = {
    r["region"]: r["prior_sales"]
    for r in df_revenue
    .filter(F.col("month") < latest_month)
    .groupBy("region")
    .agg(F.sum("total_sales").alias("prior_sales"))
    .collect()
}

# Step 4: build per-region KPI dicts with MoM delta
revenue_kpis = []
for r in revenue_current:
    region      = r["region"]
    current_val = round(float(r["total_sales"] or 0), 2)
    prior_val   = round(float(prior_by_region.get(region, current_val) or current_val), 2)
    mom_delta   = round(((current_val - prior_val) / prior_val * 100) if prior_val > 0 else 0, 1)
    revenue_kpis.append({
        "region":        region,
        "total_sales":   current_val,
        "order_count":   int(r["order_count"] or 0),
        "mom_delta_pct": mom_delta
    })

total_revenue = round(sum(r["total_sales"] for r in revenue_kpis), 2)
top_region    = revenue_kpis[0]["region"]  if revenue_kpis else "N/A"
bottom_region = revenue_kpis[-1]["region"] if revenue_kpis else "N/A"

# Step 5: serialise to JSON — this exact string is passed to the LLM
revenue_kpi_json = json.dumps({
    "period":        str(latest_month),
    "total_revenue": total_revenue,
    "top_region":    top_region,
    "bottom_region": bottom_region,
    "by_region":     revenue_kpis
})

print(f"Revenue KPIs — period (month): {latest_month}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"{'Region':<12}  {'Sales':>12}  MoM")
for r in revenue_kpis:
    print(f"  {r['region']:<10}  ${r['total_sales']:>11,.2f}  {r['mom_delta_pct']:+.1f}%")

In [0]:
# PURPOSE: Turn the 5 aggregated revenue KPIs into an executive narrative.
#
# Prompt design:
#   Opens by acknowledging the dashboard already shows these numbers.
#   Asks the model to explain the STORY behind them — root cause, not restatement.
#   Four specific questions force analytical depth:
#     1. What does the overall trend signal?
#     2. What is DRIVING the gap between top and bottom regions?
#     3. What is the most likely ROOT CAUSE of declining regions?
#     4. What should leadership prioritise in the next 30 days?

revenue_prompt = f"""Regional revenue performance for the period {latest_month} is shown below.

KPI DATA:
{revenue_kpi_json}

A dashboard already shows executives these numbers. What it does not show them is the story behind
the numbers. Write a 4-6 sentence executive synthesis that goes beyond the figures and explains:
1. What the overall revenue trend signals — is the business growing, flat, or contracting?
2. What is actually driving the gap between the top and bottom performing regions — demand
   problem, coverage problem, or momentum problem? Name the specific regions and figures.
3. For regions showing month-over-month decline, what is the most likely root cause?
4. What the leadership team should prioritise in the next 30 days.

Do not restate the numbers. Explain what they mean.
Plain business English. No bullet points. No markdown. No company name."""

revenue_summary = validate_output(call_llm(revenue_prompt, SYSTEM_MSG_EXEC))

print("\nREVENUE PERFORMANCE SUMMARY:")
print("=" * 70)
print(revenue_summary["text"])
print(f"\nllm_check: {revenue_summary['llm_check']}")

In [0]:
# PURPOSE: Compute vendor-level return rate KPIs for the LLM prompt.
#
# mv_return_rate_by_vendor column reference (from gold.py):
#   vendor_id, vendor_name, total_orders_sold, total_sales
#   total_returns, total_refund_amount, avg_refund_amount
#   return_rate_pct  <- already a PERCENTAGE (e.g. 31.0 means 31%), NOT a decimal
#
# IMPORTANT: return_rate_pct is computed in gold.py as:
#   (total_returns / total_orders_sold) * 100
# So it is already a percentage. Do NOT multiply by 100 again.
#
# We drop vendor_name from the MV before joining dim_vendors to avoid
# a duplicate column error (both tables have vendor_name).
#
# Threshold: 20% return rate is the industry benchmark for retail.
# Above 20% = active vendor management required.

vendor_agg = (
    df_vendor_rr
    .drop("vendor_name")   # drop from MV before join to avoid duplicate column
    .join(df_vendors.select("vendor_id", "vendor_name"), on="vendor_id", how="left")
    .groupBy("vendor_id", "vendor_name")
    .agg(
        F.avg("return_rate_pct")  .alias("avg_return_rate_pct"),  # already a %
        F.sum("total_returns")    .alias("total_returns"),
        F.sum("total_orders_sold").alias("total_orders")
    )
    .orderBy(F.col("avg_return_rate_pct").desc())
    .collect()
)

# Vendors above the 20% threshold need active management
high_risk_vendors = [r for r in vendor_agg if float(r["avg_return_rate_pct"] or 0) > 20.0]

avg_return_overall = round(
    sum(float(r["avg_return_rate_pct"] or 0) for r in vendor_agg) / max(len(vendor_agg), 1), 1
)

vendor_kpi_json = json.dumps({
    "total_vendors_analyzed":    len(vendor_agg),
    "vendors_above_20pct":       len(high_risk_vendors),
    "avg_return_rate_overall":   avg_return_overall,
    "top_5_highest_return_rate": [
        {
            "vendor":      r["vendor_name"] or r["vendor_id"],
            "return_rate": float(r["avg_return_rate_pct"] or 0)
        }
        for r in vendor_agg[:5]
    ]
})

print(f"Vendor return rate KPIs — {len(vendor_agg)} vendors analyzed")
print(f"Vendors above 20% threshold : {len(high_risk_vendors)}")
print(f"Fleet average return rate   : {avg_return_overall}%")
print()
print(f"{'Vendor':<30}  Return Rate")
for r in vendor_agg[:5]:
    name = str(r["vendor_name"] or r["vendor_id"])
    flag = " <-- above 20%" if float(r["avg_return_rate_pct"] or 0) > 20.0 else ""
    print(f"  {name:<28}  {r['avg_return_rate_pct']}%{flag}")

In [0]:
# PURPOSE: Explain what high return rates mean for the business — not just list them.
#
# Prompt design:
#   Asks the model to diagnose each flagged vendor:
#   - Is the high rate from product quality failure?
#   - Fulfilment or shipping defects?
#   - A category that structurally attracts returns?
#   Then asks for specific action per vendor: renegotiate, probation, or replace.

vendor_prompt = f"""Vendor return rate data across all active suppliers is shown below.

KPI DATA:
{vendor_kpi_json}

A dashboard shows these return rates as numbers. What executives need is what those numbers
mean. Write a 4-6 sentence executive synthesis that explains:
1. Whether the overall picture is a contained issue or a systemic problem.
2. For each vendor above 20%, what the rate likely indicates — product quality failure,
   fulfilment defect, description mismatch, or structurally high-return category.
   Name each vendor and its rate.
3. The financial and operational consequence of leaving high-return vendors unchanged.
4. What action to take in the next 30 days: renegotiate, probation, or alternative sourcing.

Do not restate the numbers. Explain what they mean and what to do.
Plain business English. No bullet points. No markdown. No company name."""

vendor_summary = validate_output(call_llm(vendor_prompt, SYSTEM_MSG_EXEC))

print("\nVENDOR RETURN RATE SUMMARY:")
print("=" * 70)
print(vendor_summary["text"])
print(f"\nllm_check: {vendor_summary['llm_check']}")

In [0]:
# PURPOSE: Compute slow-moving inventory KPIs for the LLM prompt.
#
# mv_slow_moving_products column reference (from gold.py):
#   product_id, product_name, brand, region
#   total_quantity_sold, total_sales, order_count
#   first_sale_date, last_sale_date
#   days_since_last_sale, selling_period_days, avg_daily_quantity
#
# Exposure estimate = SUM(total_sales) per region across slow-moving products.
# This is a conservative floor — actual cost is higher when holding cost,
# markdown risk, and shelf opportunity cost are included.

slow_agg = (
    df_slow
    .groupBy("region")
    .agg(
        F.count("*").alias("slow_product_count"),
        F.sum(
            F.when(F.col("total_sales").isNotNull(), F.col("total_sales")).otherwise(0)
        ).alias("estimated_exposure")
    )
    .orderBy(F.col("slow_product_count").desc())
    .collect()
)

total_slow     = sum(int(r["slow_product_count"] or 0) for r in slow_agg)
total_exposure = round(sum(float(r["estimated_exposure"] or 0) for r in slow_agg), 2)
worst_region   = slow_agg[0]["region"] if slow_agg else "N/A"

slow_kpi_json = json.dumps({
    "total_slow_moving_products":   total_slow,
    "total_inventory_exposure_usd": total_exposure,
    "worst_performing_region":      worst_region,
    "by_region": [
        {
            "region":       r["region"],
            "slow_count":   int(r["slow_product_count"] or 0),
            "exposure_usd": round(float(r["estimated_exposure"] or 0), 2)
        }
        for r in slow_agg
    ]
})

print("Slow-moving inventory KPIs")
print(f"Total slow products : {total_slow}")
print(f"Total exposure      : ${total_exposure:,.2f}")
print(f"Worst region        : {worst_region}")
print()
print(f"{'Region':<12}  Products  Exposure ($)")
for r in slow_agg:
    print(f"  {r['region']:<10}  {r['slow_product_count']:>8}  ${float(r['estimated_exposure'] or 0):>12,.2f}")

In [0]:
# PURPOSE: Explain WHY inventory is sitting unsold and what inaction COSTS.
#
# Prompt design:
#   Opens with: a dashboard shows counts and exposure, not why it is unsold.
#   Asks for root cause (pricing, oversupply, merchandising failure, demand mismatch)
#   and the FULL cost of inaction — holding cost, markdown risk, opportunity cost.
#   Asks for a specific resolution path per region.

slow_prompt = f"""Slow-moving inventory data by region is shown below.

KPI DATA:
{slow_kpi_json}

A dashboard shows these counts and exposure figures. What it does not show is why this inventory
is sitting unsold and what it costs to leave it there. Write a 4-6 sentence executive synthesis:
1. What the pattern reveals — localised problem or broader demand forecasting failure?
2. For the highest-concentration region, what is the most likely cause — pricing, vendor
   oversupply, merchandising failure, or demand mismatch?
3. The full cost of inaction — not just the exposure figure but holding cost, markdown risk,
   and the opportunity cost of tied-up shelf space.
4. The most appropriate resolution — clearance pricing, inter-region transfer, vendor buyback,
   or discontinuation.

Do not restate the numbers. Explain what they mean and why the situation exists.
Plain business English. No bullet points. No markdown. No company name."""

slow_summary = validate_output(call_llm(slow_prompt, SYSTEM_MSG_EXEC))

print("\nSLOW-MOVING INVENTORY SUMMARY:")
print("=" * 70)
print(slow_summary["text"])
print(f"\nllm_check: {slow_summary['llm_check']}")

In [0]:
# PURPOSE: Persist all three executive summaries to globalmart.gold.ai_business_insights.
#
# Output schema:
#   insight_type     — domain: revenue_performance, vendor_return_rate, slow_moving_inventory
#   ai_summary       — the 4-6 sentence synthesis
#   kpi_json         — the exact KPI JSON passed to the LLM (for auditability)
#   llm_check        — PASS or REVIEW
#   llm_check_detail — list of issues found, or 'none'
#   generated_at     — ISO timestamp
#
# Write mode = APPEND: each run adds rows rather than overwriting.
# This preserves history so summaries can be compared across periods.
# kpi_json is stored so any analyst can verify what data produced each summary.

insights = [
    {
        "insight_type":     "revenue_performance",
        "ai_summary":       revenue_summary["text"],
        "kpi_json":         revenue_kpi_json,
        "llm_check":        revenue_summary["llm_check"],
        "llm_check_detail": revenue_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    },
    {
        "insight_type":     "vendor_return_rate",
        "ai_summary":       vendor_summary["text"],
        "kpi_json":         vendor_kpi_json,
        "llm_check":        vendor_summary["llm_check"],
        "llm_check_detail": vendor_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    },
    {
        "insight_type":     "slow_moving_inventory",
        "ai_summary":       slow_summary["text"],
        "kpi_json":         slow_kpi_json,
        "llm_check":        slow_summary["llm_check"],
        "llm_check_detail": slow_summary["llm_check_detail"],
        "generated_at":     datetime.now().isoformat()
    }
]

schema = StructType([
    StructField("insight_type",     StringType(), True),
    StructField("ai_summary",       StringType(), True),
    StructField("kpi_json",         StringType(), True),
    StructField("llm_check",        StringType(), True),
    StructField("llm_check_detail", StringType(), True),
    StructField("generated_at",     StringType(), True)
])

df_insights = spark.createDataFrame(insights, schema=schema)

(
    df_insights.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.ai_business_insights")
)

# OPTIMIZE after write — dashboard queries filter on insight_type and generated_at
spark.sql(f"OPTIMIZE {CATALOG}.ai_business_insights ZORDER BY (insight_type, generated_at)")

print(f"Written to {CATALOG}.ai_business_insights")
print("OPTIMIZE + ZORDER complete")
display(spark.table(f"{CATALOG}.ai_business_insights").orderBy(F.col("generated_at").desc()).limit(3))

In [0]:
# PURPOSE: Demonstrate Databricks ai_query() — calls a Foundation Model
# directly inside a SQL query. No Python API client needed.
#
# HOW THE RESPONSE EXTRACTION WORKS:
# databricks-gpt-oss-20b returns a JSON array:
#   [{"type": "reasoning", ...},        <- internal thinking, discard
#    {"type": "text", "text": "..."}]   <- the answer, keep this
#
# Two get_json_object() calls extract the answer:
#   get_json_object(ai_query(...), '$[1]')    -> gets the text block object
#   get_json_object(above,         '$.text') -> gets the answer string
#
# DEMO 1: vendor return rate interpretation — one sentence per vendor
# DEMO 2: revenue trend commentary — one sentence per region
#
# Note: return_rate_pct in mv_return_rate_by_vendor is already a percentage
# (e.g. 31.0 = 31%). Do NOT multiply by 100 again.

print("=" * 70)
print("DEMO 1 — ai_query() on vendor return rates")
print("Reads: globalmart.gold.mv_return_rate_by_vendor")
print("=" * 70)

spark.sql(f"""
SELECT
    vendor_id,
    vendor_name,
    ROUND(return_rate_pct, 1)                       AS return_rate_pct,
    get_json_object(
        get_json_object(
            ai_query(
                '{MODEL_NAME}',
                CONCAT(
                    'Vendor ', vendor_name,
                    ' has a return rate of ', CAST(ROUND(return_rate_pct, 1) AS STRING), '%. ',
                    'In one sentence, assess whether this is acceptable (under 15%), ',
                    'concerning (15-25%), or critical (above 25%) for a national retail chain, ',
                    'and state the most likely business reason: product quality, ',
                    'fulfilment error, or category mismatch. Plain English only. No markdown.'
                )
            ),
            '$[1]'
        ),
        '$.text'
    )                                               AS risk_assessment
FROM {CATALOG}.mv_return_rate_by_vendor
ORDER BY return_rate_pct DESC
LIMIT 5
""").display()

print()
print("=" * 70)
print("DEMO 2 — ai_query() on revenue by region (latest month)")
print("Reads: globalmart.gold.mv_revenue_by_region")
print("=" * 70)

spark.sql(f"""
SELECT
    region,
    ROUND(total_sales, 0)                           AS total_sales,
    order_count,
    get_json_object(
        get_json_object(
            ai_query(
                '{MODEL_NAME}',
                CONCAT(
                    'The ', region, ' region generated $',
                    CAST(ROUND(total_sales, 0) AS STRING),
                    ' across ', CAST(order_count AS STRING), ' orders this period. ',
                    'In one sentence, state whether this signals growth, stability, or concern ',
                    'and identify one likely driver. Plain English only. No markdown.'
                )
            ),
            '$[1]'
        ),
        '$.text'
    )                                               AS revenue_commentary
FROM {CATALOG}.mv_revenue_by_region
WHERE month = (SELECT MAX(month) FROM {CATALOG}.mv_revenue_by_region)
ORDER BY total_sales DESC
""").display()

---
## Part B — AI Lead Scoring

**What this section does:**
Scores every customer on four components (total 0-100 pts), classifies them as
HOT / WARM / COLD, then uses `ai_query()` to generate a one-sentence SHAP-style
explanation per customer naming the single biggest factor that drove their score.

**Score components:**

| Component | Max pts | What it measures |
|---|---|---|
| purchase_value_score | 40 | Lifetime spend vs fleet maximum (sqrt normalised) |
| frequency_score | 30 | Order count vs fleet average (capped at 30) |
| recency_score | 20 | Ordered in last 180 days = 20 pts, else 0 |
| return_penalty | -10 | Deducted when return rate > 30% |

**Labels:**
- HOT >= 65 — high value, active, low return risk
- WARM 35-64 — moderate engagement, worth nurturing
- COLD < 35 — low engagement or high return risk

**SHAP explanation:**
The `ai_query()` call in Cell 15 runs one LLM call per customer row inside SQL.
It names the single biggest driver so the sales team knows exactly what to act on.

**Output table:** `globalmart.gold.customer_lead_scores`

In [0]:
# PURPOSE: Build one profile row per customer from fact_sales and fact_returns.
#
# fact_sales column reference (from gold.py):
#   customer_id, order_id, product_id, vendor_id
#   sales, quantity, profit, discount
#   order_date_key  <- integer in yyyyMMdd format, e.g. 20180315
#
# fact_returns column reference (from gold.py):
#   customer_id, order_id, refund_amount
#   return_to_sales_ratio  <- refund_amount / original_sales
#
# order_date_key is converted to a date for the datediff recency calculation.
# return_rate = return_count / total_orders — always between 0.0 and 1.0.
#
# DESIGN RULE: aggregate first, never pass raw rows to the LLM.

# Step 1: purchase signals from fact_sales
df_purchase = (
    df_sales
    .groupBy("customer_id")
    .agg(
        F.round(F.sum("sales"), 2)       .alias("total_spent"),
        F.countDistinct("order_id")      .alias("total_orders"),
        F.round(F.avg("sales"), 2)       .alias("avg_order_value"),
        F.max("order_date_key")          .alias("last_order_date_key")
    )
    # Convert yyyyMMdd integer to a date, then compute days since last order
    .withColumn(
        "last_order_date",
        F.to_date(F.col("last_order_date_key").cast("string"), "yyyyMMdd")
    )
    .withColumn(
        "days_since_last_order",
        F.datediff(F.current_date(), F.col("last_order_date"))
    )
    # 1 if ordered in last 180 days (active customer), 0 if not (lapsed)
    .withColumn(
        "has_recent_order",
        F.when(F.col("days_since_last_order") <= 180, 1).otherwise(0)
    )
)

# Step 2: return signals from fact_returns
df_return_signals = (
    df_returns
    .groupBy("customer_id")
    .agg(
        F.count("*")                       .alias("return_count"),
        F.round(F.sum("refund_amount"), 2) .alias("total_refunded")
    )
)

# Step 3: join into one profile per customer, add customer name/segment/region
df_profiles = (
    df_purchase
    .join(df_return_signals, on="customer_id", how="left")
    .join(
        df_customers.select("customer_id", "customer_name", "segment", "region"),
        on="customer_id", how="left"
    )
    .fillna({"return_count": 0, "total_refunded": 0.0})
    # return_rate = return_count / total_orders — always 0.0 to 1.0
    .withColumn(
        "return_rate",
        F.when(F.col("total_orders") > 0,
               F.round(F.col("return_count") / F.col("total_orders"), 3))
         .otherwise(F.lit(0.0))
    )
)

print(f"Customer profiles built: {df_profiles.count():,}")
display(
    df_profiles
    .select("customer_id", "customer_name", "segment", "region",
            "total_spent", "total_orders", "days_since_last_order",
            "return_count", "return_rate")
    .orderBy(F.col("total_spent").desc())
    .limit(10)
)

In [0]:
# PURPOSE: Apply the 4-component scoring model to every customer profile.
#
# Normalisation approach:
#   purchase_value_score uses sqrt(total_spent) / sqrt(fleet_max) * 40
#   Reason: linear normalisation would compress all customers near zero if
#   one outlier spent 10x more than everyone else. Sqrt compresses the range
#   while preserving the ordering — high spenders still score higher.
#
#   frequency_score uses (total_orders / fleet_avg) * 15, capped at 30
#   Reason: a customer at exactly the fleet average gets 15 pts (half of max).
#   Cap at 30 prevents one very frequent buyer from dominating the score.
#
# Fleet statistics are computed once as Python floats and passed as literals.
# This avoids recomputing them inside every withColumn call.

max_spent  = float(df_profiles.agg(F.max("total_spent")) .collect()[0][0] or 1.0)
avg_orders = float(df_profiles.agg(F.avg("total_orders")).collect()[0][0] or 1.0)

print(f"Fleet max spend  : ${max_spent:,.2f}")
print(f"Fleet avg orders : {avg_orders:.1f}")

df_scored = (
    df_profiles

    # Component 1 — purchase value (0 to 40 pts)
    # sqrt normalisation prevents extreme outliers from compressing others near zero
    .withColumn(
        "purchase_value_score",
        F.round(
            F.least(
                F.lit(40.0),
                (F.sqrt(F.col("total_spent")) / F.sqrt(F.lit(max_spent))) * 40.0
            ), 1
        )
    )

    # Component 2 — purchase frequency (0 to 30 pts)
    # customer at fleet average gets 15 pts; cap at 30
    .withColumn(
        "frequency_score",
        F.round(
            F.least(
                F.lit(30.0),
                (F.col("total_orders") / F.lit(avg_orders)) * 15.0
            ), 1
        )
    )

    # Component 3 — recency (0 or 20 pts)
    # ordered in last 180 days = active customer = full 20 pts
    .withColumn(
        "recency_score",
        (F.col("has_recent_order") * 20).cast("double")
    )

    # Component 4 — return penalty (0 to 10 pts, subtracted)
    # only activates when return rate exceeds 30%
    # formula: MIN(10, MAX(0, (return_rate - 0.30) * 50))
    # 30% rate = 0 penalty | 50% rate = 10 penalty
    .withColumn(
        "return_penalty",
        F.round(
            F.least(
                F.lit(10.0),
                F.greatest(
                    F.lit(0.0),
                    (F.col("return_rate") - F.lit(0.30)) * 50.0
                )
            ), 1
        )
    )

    # Total lead score
    .withColumn(
        "lead_score",
        F.round(
            F.col("purchase_value_score") +
            F.col("frequency_score") +
            F.col("recency_score") -
            F.col("return_penalty"),
            1
        )
    )

    # Label — HOT >= 65, WARM >= 35, COLD < 35
    .withColumn(
        "lead_label",
        F.when(F.col("lead_score") >= 65, "HOT")
         .when(F.col("lead_score") >= 35, "WARM")
         .otherwise("COLD")
    )
)

print("\nLabel distribution:")
df_scored.groupBy("lead_label").count().orderBy("lead_label").show()

print("Top 20 customers by lead score:")
display(
    df_scored
    .select("customer_id", "customer_name", "segment", "region",
            "total_spent", "total_orders", "return_rate",
            "purchase_value_score", "frequency_score",
            "recency_score", "return_penalty",
            "lead_score", "lead_label")
    .orderBy(F.col("lead_score").desc())
    .limit(20)
)

In [0]:
# PURPOSE: Generate a one-sentence SHAP-style explanation per customer using ai_query().
#
# Why ai_query() here (not the Python OpenAI client):
#   ai_query() runs inside SQL — one LLM call per row, no Python loop needed.
#   This is the correct Databricks pattern for per-row LLM inference.
#   It also demonstrates the SQL-native AI capability required by the brief.
#
# What the SHAP explanation does:
#   The prompt gives all four score components and asks the model to name the
#   SINGLE BIGGEST FACTOR driving the score. This is equivalent to a SHAP
#   top-feature explanation — it tells the sales team exactly what to act on.
#
# Response extraction:
#   databricks-gpt-oss-20b returns a JSON array:
#     [{"type": "reasoning", ...},        <- internal thinking, discard
#      {"type": "text", "text": "..."}]   <- the answer, keep
#   get_json_object(result, '$[1]')       -> text block
#   get_json_object(text_block, '$.text') -> answer string
#
# LIMIT 100: processes the top 100 customers by score. Remove for all customers.

# Write to a temp view so ai_query() can read the scored data in SQL
df_scored.createOrReplaceTempView("v_lead_scores")

print("Generating SHAP explanations via ai_query()...")
print("One LLM call per customer row inside SQL.")
print("Processing top 100 customers by lead score.\n")

df_lead_scores = spark.sql(f"""
SELECT
    customer_id,
    customer_name,
    segment,
    region,
    ROUND(total_spent, 2)                AS total_spent,
    total_orders,
    ROUND(return_rate * 100, 1)          AS return_rate_pct,
    days_since_last_order,
    ROUND(purchase_value_score, 1)       AS purchase_value_score,
    ROUND(frequency_score, 1)            AS frequency_score,
    ROUND(recency_score, 1)              AS recency_score,
    ROUND(return_penalty, 1)             AS return_penalty,
    ROUND(lead_score, 1)                 AS lead_score,
    lead_label,

    -- SHAP-style explanation: one sentence naming the top driver
    get_json_object(
        get_json_object(
            ai_query(
                '{MODEL_NAME}',
                CONCAT(
                    'A retail customer has a lead score of ',
                    CAST(ROUND(lead_score, 0) AS STRING),
                    ' out of 100 (label: ', lead_label, '). ',
                    'Score components: ',
                    'purchase value = ', CAST(ROUND(purchase_value_score, 1) AS STRING), ' pts, ',
                    'order frequency = ', CAST(ROUND(frequency_score, 1) AS STRING), ' pts, ',
                    'recency = ', CAST(ROUND(recency_score, 1) AS STRING), ' pts, ',
                    'return penalty = -', CAST(ROUND(return_penalty, 1) AS STRING), ' pts. ',
                    'Segment: ', COALESCE(segment, 'unknown'), '. ',
                    'Region: ', COALESCE(region, 'unknown'), '. ',
                    'In one sentence, name the single biggest factor that drove this score ',
                    '(purchase value, order frequency, recency, or return behaviour) ',
                    'and state what the sales team should do next for a ', lead_label, ' customer. ',
                    'Plain English only. No markdown.'
                )
            ),
            '$[1]'
        ),
        '$.text'
    )                                    AS shap_explanation

FROM v_lead_scores
ORDER BY lead_score DESC
LIMIT 100
""")

print("Explanations generated. Sample output:")
display(df_lead_scores.limit(10))

In [0]:
# PURPOSE: Write lead scores to globalmart.gold.customer_lead_scores.
#
# Write mode = OVERWRITE:
#   Lead scores are a current-state view, not historical audit data.
#   Each run replaces the previous scores entirely — unlike ai_business_insights
#   which appends so summaries can be compared across periods.
#
# OPTIMIZE + ZORDER:
#   Dashboard queries filter on lead_label and sort by lead_score descending.
#   ZORDER on both eliminates full table scans for those query patterns.

(
    df_lead_scores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.customer_lead_scores")
)

spark.sql(f"OPTIMIZE {CATALOG}.customer_lead_scores ZORDER BY (lead_label, lead_score)")

print(f"Written to {CATALOG}.customer_lead_scores")
print("OPTIMIZE + ZORDER complete")

# Label counts — used by the run log cell below
total_scored = df_lead_scores.count()
hot_count    = df_lead_scores.filter(F.col("lead_label") == "HOT") .count()
warm_count   = df_lead_scores.filter(F.col("lead_label") == "WARM").count()
cold_count   = df_lead_scores.filter(F.col("lead_label") == "COLD").count()

print(f"\nTotal scored : {total_scored:,}")
print(f"HOT          : {hot_count:,}")
print(f"WARM         : {warm_count:,}")
print(f"COLD         : {cold_count:,}")

# Print one representative example per label — for the hackathon screenshot
print("\n" + "=" * 70)
print("SAMPLE SHAP EXPLANATIONS BY LABEL")
print("=" * 70)
for label in ["HOT", "WARM", "COLD"]:
    row = df_lead_scores.filter(F.col("lead_label") == label) \
                        .orderBy(F.col("lead_score").desc()) \
                        .limit(1).collect()
    if row:
        r = row[0]
        print(f"\n[{label}] {r['customer_name']} | Score: {r['lead_score']} | "
              f"Segment: {r['segment']} | Region: {r['region']}")
        print(f"  value={r['purchase_value_score']}  freq={r['frequency_score']}  "
              f"recency={r['recency_score']}  penalty=-{r['return_penalty']}")
        print(f"  SHAP: {r['shap_explanation']}")

display(spark.table(f"{CATALOG}.customer_lead_scores").orderBy(F.col("lead_score").desc()))

In [0]:
# PURPOSE: Append one row to the pipeline run log.
#
# Captures:
#   - When this notebook ran
#   - How many records were processed (3 executive summaries + lead scores)
#   - How many LLM calls were made (3 Python calls + N ai_query SQL calls)
#   - Whether any executive summaries were flagged for review
#   - Lead scoring label counts for operational monitoring
#
# Status = PARTIAL_REVIEW if any of the 3 executive summaries failed quality
# validation. Lead scoring via ai_query() does not contribute to this flag.

review_count = sum(
    1 for s in [revenue_summary, vendor_summary, slow_summary]
    if s["llm_check"] == "REVIEW"
)

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": 3 + total_scored,
    "llm_calls_made":    3 + total_scored,   # 3 Python + N ai_query SQL calls
    "outputs_flagged":   review_count,
    "status":            "SUCCESS" if review_count == 0 else "PARTIAL_REVIEW",
    "notes": (
        f"domains=revenue,vendor_return_rate,slow_inventory | period={latest_month} | "
        f"lead_scores={total_scored} | hot={hot_count} | warm={warm_count} | cold={cold_count}"
    )
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.pipeline_run_log")
)

print(f"Run log written. Status: {run_log[0]['status']}")
print(f"Executive summaries  : 3 domains")
print(f"Review flags         : {review_count}")
print(f"Lead scores written  : {total_scored}")
print(f"  HOT                : {hot_count}")
print(f"  WARM               : {warm_count}")
print(f"  COLD               : {cold_count}")